# CustomerPulse - Data Quality Assessment

In [1]:
import pandas as pd

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("Libraries loaded successfully.")

Libraries loaded successfully.


# Load Datasets

In [2]:
import pandas as pd

customers = pd.read_csv("../Data/Raw/olist_customers_dataset.csv")
geolocation = pd.read_csv("../Data/Raw/olist_geolocation_dataset.csv")
order_items = pd.read_csv("../Data/Raw/olist_order_items_dataset.csv")
payments = pd.read_csv("../Data/Raw/olist_order_payments_dataset.csv")
reviews = pd.read_csv("../Data/Raw/olist_order_reviews_dataset.csv")
orders = pd.read_csv("../Data/Raw/olist_orders_dataset.csv")
products = pd.read_csv("../Data/Raw/olist_products_dataset.csv")
sellers = pd.read_csv("../Data/Raw/olist_sellers_dataset.csv")
category_translation = pd.read_csv("../Data/Raw/product_category_name_translation.csv")

print("All datasets loaded successfully.")

All datasets loaded successfully.


In [3]:
datasets = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}

# Data Quality Assessment

In [4]:
for name, df in datasets.items():

    print("\n" + "="*70)
    print(f"TABLE: {name.upper()}")
    print("="*70)

    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")

    print("\nData Types:")
    print(df.dtypes)

    print("\nMissing Values:")
    print(df.isnull().sum())

    print("\nDuplicate Rows:")
    print(df.duplicated().sum())


TABLE: CUSTOMERS
Rows: 99441
Columns: 5

Data Types:
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

Missing Values:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Duplicate Rows:
0

TABLE: GEOLOCATION
Rows: 1000163
Columns: 5

Data Types:
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object

Missing Values:
geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

Duplicate Rows:
261831

TABLE: ORDER_ITEMS
Rows: 112650
Columns: 7

Data Types:
order_id       

# Duplicate Investigation

In [5]:
geolocation.duplicated().sum()

261831

In [6]:
geo_clean = geolocation.drop_duplicates()

print("Original Rows:", len(geolocation))
print("Cleaned Rows:", len(geo_clean))
print("Removed:", len(geolocation) - len(geo_clean))

Original Rows: 1000163
Cleaned Rows: 738332
Removed: 261831


# Missing Value Investigation

In [7]:
products.isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [8]:
products[products.isnull().any(axis=1)].head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0


In [9]:
products[products["product_category_name"].isnull()].shape

(610, 9)

# Cleaning Actions

In [10]:
products[
    products["product_weight_g"].isnull()
]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
# Replace missing categories
products["product_category_name"] = (
    products["product_category_name"]
    .fillna("Unknown")
)

In [12]:
products["product_category_name"].isnull().sum()

0

In [13]:
geo_clean = geolocation.drop_duplicates()

In [14]:
geo_clean.duplicated().sum()

0

In [15]:
geo_clean.to_csv(
    "../Data/Cleaned/olist_geolocation_dataset.csv",
    index=False
)

products.to_csv(
    "../Data/Cleaned/olist_products_dataset.csv",
    index=False
)

# Datetime Conversion

In [16]:
orders.dtypes

order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

In [17]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(
        orders[col],
        errors="coerce"
    )

In [18]:
orders[date_columns].dtypes

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

# Additional Date Columns Investigation

In [20]:
reviews.dtypes

review_id                  object
order_id                   object
review_score                int64
review_comment_title       object
review_comment_message     object
review_creation_date       object
review_answer_timestamp    object
dtype: object

In [21]:
reviews.dtypes

review_id                  object
order_id                   object
review_score                int64
review_comment_title       object
review_comment_message     object
review_creation_date       object
review_answer_timestamp    object
dtype: object

# Additional Datetime Conversion

In [22]:
# Convert review dates

review_date_columns = [
    "review_creation_date",
    "review_answer_timestamp"
]

for col in review_date_columns:
    reviews[col] = pd.to_datetime(
        reviews[col],
        errors="coerce"
    )

In [23]:
reviews[review_date_columns].dtypes

review_creation_date       datetime64[ns]
review_answer_timestamp    datetime64[ns]
dtype: object

In [24]:
# Convert shipping date

order_items["shipping_limit_date"] = pd.to_datetime(
    order_items["shipping_limit_date"],
    errors="coerce"
)

In [25]:
order_items["shipping_limit_date"].dtype

dtype('<M8[ns]')

# Export Cleaned Datasets

In [26]:
# Save cleaned datasets

customers.to_csv(
    "../Data/Cleaned/olist_customers_dataset.csv",
    index=False
)

geo_clean.to_csv(
    "../Data/Cleaned/olist_geolocation_dataset.csv",
    index=False
)

order_items.to_csv(
    "../Data/Cleaned/olist_order_items_dataset.csv",
    index=False
)

payments.to_csv(
    "../Data/Cleaned/olist_order_payments_dataset.csv",
    index=False
)

reviews.to_csv(
    "../Data/Cleaned/olist_order_reviews_dataset.csv",
    index=False
)

orders.to_csv(
    "../Data/Cleaned/olist_orders_dataset.csv",
    index=False
)

products.to_csv(
    "../Data/Cleaned/olist_products_dataset.csv",
    index=False
)

sellers.to_csv(
    "../Data/Cleaned/olist_sellers_dataset.csv",
    index=False
)

category_translation.to_csv(
    "../Data/Cleaned/product_category_name_translation.csv",
    index=False
)

print("All cleaned datasets exported successfully.")

All cleaned datasets exported successfully.


In [27]:
import os

cleaned_files = os.listdir("../Data/Cleaned")

print("Files exported:", len(cleaned_files))
print(cleaned_files)

Files exported: 9
['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_orders_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']
